In [1]:
import re, json
import numpy as np
from collections import defaultdict, Counter
import matplotlib; matplotlib.use('Agg')
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

In [2]:
SEED = 42
np.random.seed(SEED)


In [3]:
def load_corpus(path):
    with open(path, encoding='utf-8') as f:
        content = f.read()
    parts = re.split(r'\[(\d+)\]', content)
    docs = {}
    for i in range(1, len(parts)-1, 2):
        docs[int(parts[i])] = parts[i+1].strip()
    return docs


In [5]:
def tokenize(text): return [t for t in text.split() if t]
 
docs_clean = load_corpus('cleaned.txt')
doc_ids    = sorted(docs_clean.keys())
N          = len(doc_ids)
 
all_tokens = []
for did in doc_ids: all_tokens.extend(tokenize(docs_clean[did]))
global_freq = Counter(all_tokens)
 
VOCAB_SIZE = 10_000
top_vocab  = [w for w, _ in global_freq.most_common(VOCAB_SIZE)]
word2idx   = {w: i for i, w in enumerate(top_vocab)}
word2idx['<UNK>'] = VOCAB_SIZE
UNK_IDX = VOCAB_SIZE
idx2word = {v: k for k, v in word2idx.items()}
 
print(f"Vocab size: {VOCAB_SIZE}, Total tokens: {len(all_tokens):,}")
 
CATEGORIES = {
    'Politics':         ['حکومت','وزیر','پارلیمنٹ','انتخاب','سیاس','جماعت','تحریک','عدالت','سپریم','قانون'],
    'Sports':           ['کرکٹ','میچ','ٹیم','کھلاڑ','اسکور','فٹبال','کھیل','ٹورنامنٹ','وکٹ','رن'],
    'Economy':          ['مہنگا','تجارت','بینک','بجٹ','روپیہ','مالی','اقتصاد','افراط','قیمت','معیشت'],
    'International':    ['اقوام','معاہد','سفارت','تنازع','امریک','بھارت','چین','روس','عالم'],
    'Health & Society': ['ہسپتال','بیمار','ویکسین','سیلاب','تعلیم','صحت','وباء','علاج','ڈاکٹر'],
}


Vocab size: 10000, Total tokens: 362,530


In [6]:
def assign_category(text):
    tokens = set(tokenize(text))
    scores = {c: sum(1 for kw in kws if kw in tokens) for c, kws in CATEGORIES.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'Politics'
 
doc_categories = {did: assign_category(docs_clean[did]) for did in doc_ids}
cat_docs_idx   = defaultdict(list)
for d_i, did in enumerate(doc_ids):
    cat_docs_idx[doc_categories[did]].append(d_i)
 
cat_counts = Counter(doc_categories.values())
print("Category distribution:")
for cat, cnt in cat_counts.most_common():
    print(f"  {cat:<25}: {cnt}")


Category distribution:
  Politics                 : 165
  Sports                   : 28
  International            : 28
  Health & Society         : 23
  Economy                  : 6


In [9]:
D = VOCAB_SIZE + 1
tf_matrix = np.zeros((N, D), dtype=np.float32)
for d_i, did in enumerate(doc_ids):
    tokens = tokenize(docs_clean[did])
    total  = max(len(tokens), 1)
    for w, cnt in Counter(tokens).items():
        tf_matrix[d_i, word2idx.get(w, UNK_IDX)] += cnt / total
 
df  = (tf_matrix > 0).sum(axis=0).astype(np.float32)
idf = np.log(N / (1 + df)).astype(np.float32)
tfidf_matrix = (tf_matrix * idf[np.newaxis, :]).astype(np.float32)
 
np.save('tfidf_matrix.npy', tfidf_matrix)
print(f"\nTF-IDF matrix: {tfidf_matrix.shape}  saved ")
 
print("\nTop-10 discriminative words per category:")
for cat in CATEGORIES:
    if cat not in cat_docs_idx: continue
    idxs  = cat_docs_idx[cat]
    avg   = tfidf_matrix[idxs, :].mean(axis=0)
    top10 = np.argsort(avg)[::-1][:10]
    print(f"\n[{cat}]")
    for r, wi in enumerate(top10, 1):
        print(f"  {r:2d}. {idx2word.get(wi,'<UNK>'):<22} {avg[wi]:.5f}")



TF-IDF matrix: (250, 10001)  saved 

Top-10 discriminative words per category:

[Politics]
   1. <UNK>                  0.00312
   2. روس                    0.00288
   3. پولیس                  0.00265
   4. ایپسٹین                0.00249
   5. خان                    0.00232
   6. فوج                    0.00216
   7. انڈ                    0.00201
   8. حکومت                  0.00185
   9. کینیڈ                  0.00181
  10. سنگھ                   0.00180

[Sports]
   1. میچ                    0.01107
   2. کرکٹ                   0.01016
   3. کھلاڑ                  0.00826
   4. کپ                     0.00701
   5. ورلڈ                   0.00544
   6. ٹیم                    0.00523
   7. کھیل                   0.00504
   8. دیش                    0.00481
   9. بل                     0.00476
  10. کھیلن                  0.00459

[Economy]
   1. فلم                    0.01758
   2. ضوفش                   0.01064
   3. صارفین                 0.01041
   4. دیش                    0.00967

In [10]:
meta = {'top_vocab': top_vocab, 'word2idx': word2idx,
        'doc_categories': {str(k):v for k,v in doc_categories.items()},
        'cat_docs_idx': {k: list(v) for k, v in cat_docs_idx.items()}}
with open('Metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False)
print("\nmeta.json saved")
print("Stage 1 DONE")
"""Stage 2: PPMI, t-SNE, Nearest Neighbours"""



meta.json saved
Stage 1 DONE


'Stage 2: PPMI, t-SNE, Nearest Neighbours'

In [11]:
import re, json
import numpy as np
from collections import defaultdict
from sklearn.manifold import TSNE
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings; warnings.filterwarnings('ignore')


In [12]:
def load_corpus(path):
    with open(path, encoding='utf-8') as f: content = f.read()
    parts = re.split(r'\[(\d+)\]', content)
    docs = {}
    for i in range(1, len(parts)-1, 2): docs[int(parts[i])] = parts[i+1].strip()
    return docs


In [13]:
def tokenize(text): return [t for t in text.split() if t]
 
docs_clean = load_corpus('cleaned.txt')
doc_ids    = sorted(docs_clean.keys())
 
with open('Metadata.json', encoding='utf-8') as f:
    meta = json.load(f)
top_vocab     = meta['top_vocab']
word2idx      = meta['word2idx']
cat_docs_idx  = {k: v for k, v in meta['cat_docs_idx'].items()}
VOCAB_SIZE    = 10_000
 
tfidf_matrix = np.load('tfidf_matrix.npy')


In [14]:
# ── PPMI ──────────────────────────────────────────────────────
K_WIN   = 5
print(f"Building co-occurrence matrix (window={K_WIN}) ...")
cooccur = np.zeros((VOCAB_SIZE, VOCAB_SIZE), dtype=np.float32)
for did in doc_ids:
    idxs = [word2idx[t] for t in tokenize(docs_clean[did])
            if t in word2idx and word2idx[t] < VOCAB_SIZE]
    for i, ci in enumerate(idxs):
        for j in range(max(0, i-K_WIN), min(len(idxs), i+K_WIN+1)):
            if i != j: cooccur[ci, idxs[j]] += 1
 
print(f"Non-zero: {np.count_nonzero(cooccur):,}")
 
total = cooccur.sum()
wp    = cooccur.sum(axis=1) / total
cp    = cooccur.sum(axis=0) / total
denom = np.outer(wp, cp)
with np.errstate(divide='ignore', invalid='ignore'):
    joint = cooccur / total
    pmi   = np.log2(np.where(joint > 0, joint / np.maximum(denom, 1e-30), 1e-30))
ppmi_matrix = np.maximum(0, pmi).astype(np.float32)
np.fill_diagonal(ppmi_matrix, 0)
del cooccur, joint, pmi, denom  # free RAM
 
np.save('ppmi_matrix.npy', ppmi_matrix)
print(f"PPMI matrix: {ppmi_matrix.shape}  non-zero: {np.count_nonzero(ppmi_matrix):,}  saved")


Building co-occurrence matrix (window=5) ...
Non-zero: 929,690
PPMI matrix: (10000, 10000)  non-zero: 785,260  saved


In [15]:
# ── t-SNE ─────────────────────────────────────────────────────
CAT_COL = {
    'Politics':'#e74c3c','Sports':'#2ecc71','Economy':'#3498db',
    'International':'#9b59b6','Health & Society':'#f39c12',
}
TOP_N = 200
top200_vec = ppmi_matrix[:TOP_N, :]


In [17]:
# Label each word by highest avg TF-IDF category
word_cat = []
for wi in range(TOP_N):
    best, bscore = None, -1
    for cat, didxs in cat_docs_idx.items():
        sc = tfidf_matrix[didxs, wi].mean()
        if sc > bscore: bscore, best = sc, cat
    word_cat.append(best or 'Politics')
 
print("Running t-SNE ...")

tsne   = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
coords = tsne.fit_transform(top200_vec)
 
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_facecolor('#1a1a2e'); fig.patch.set_facecolor('#16213e')
for i in range(TOP_N):
    col = CAT_COL.get(word_cat[i], '#fff')
    ax.scatter(coords[i,0], coords[i,1], c=col, s=50, alpha=0.85, zorder=2)
    ax.annotate(top_vocab[i], (coords[i,0], coords[i,1]),
                fontsize=6.5, color=col, alpha=0.9, xytext=(3,3),
                textcoords='offset points')
patches = [mpatches.Patch(color=c, label=cat) for cat,c in CAT_COL.items()]
ax.legend(handles=patches, loc='upper left', fontsize=10, framealpha=0.3,
          facecolor='#2d2d2d', edgecolor='white', labelcolor='white')
ax.set_title('t-SNE of Top-200 PPMI Vectors — BBC Urdu Corpus', fontsize=14, color='white', pad=15)
ax.set_xlabel('t-SNE Dimension 1', color='white', fontsize=11)
ax.set_ylabel('t-SNE Dimension 2', color='white', fontsize=11)
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')
plt.tight_layout()
plt.savefig('tsne_ppmi.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close()
print("tsne_ppmi.png saved")



Running t-SNE ...
tsne_ppmi.png saved


In [18]:
# ── Nearest Neighbours ────────────────────────────────────────
norms = np.linalg.norm(ppmi_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1e-9
ppmi_norm = ppmi_matrix / norms
 
QUERY_WORDS = ['حکومت','پاکست','عدالت','فوج','معیشت','کرکٹ','صحت','تعلیم','انتخاب','وزیر']
print("\nTop-5 Nearest Neighbours  PPMI cosine similarity")
print("=" * 55)
for qw in QUERY_WORDS:
    if qw not in word2idx or word2idx[qw] >= VOCAB_SIZE:
        print(f"'{qw}' — not in vocab"); continue
    qi   = word2idx[qw]
    sims = ppmi_norm[qi] @ ppmi_norm.T; sims[qi] = -1
    top5 = np.argsort(sims)[::-1][:5]
    print(f"\nQuery: '{qw}'")
    for r, wi in enumerate(top5, 1):
        print(f"  {r}. {top_vocab[wi]:<22} sim={sims[wi]:.4f}")
 
# Save normed ppmi for MRR later
np.save('ppmi_norm.npy', ppmi_norm)
print("\nppmi_norm.npy saved ")
print("Stage 2 DONE")



Top-5 Nearest Neighbours  PPMI cosine similarity

Query: 'حکومت'
  1. صوبا                   sim=0.1864
  2. وفاق                   sim=0.1554
  3. کے                     sim=0.1415
  4. تحریک                  sim=0.1354
  5. پختونخو                sim=0.1350

Query: 'پاکست'
  1. انڈ                    sim=0.1919
  2. کرکٹ                   sim=0.1699
  3. کے                     sim=0.1593
  4. میچ                    sim=0.1465
  5. نے                     sim=0.1448

Query: 'عدالت'
  1. جج                     sim=0.2443
  2. سماعت                  sim=0.2285
  3. چٹھ                    sim=0.2119
  4. کورٹ                   sim=0.2065
  5. ہائیکورٹ               sim=0.2042

Query: 'فوج'
  1. افسر                   sim=0.1933
  2. جنگ                    sim=0.1914
  3. یوکرین                 sim=0.1864
  4. انڈین                  sim=0.1677
  5. روس                    sim=0.1550

Query: 'معیشت'
  1. لڑکھڑ                  sim=0.2169
  2. بسواجیت                sim=0.1946
  3. امتزاج   